# Recommendation detection: prepare data

Prepare the shared recommendation dataset using the original cleaning, sampling and preprocessing steps. Exclude ambiguous `no_idea` labels from this binary experiment; keep the source file for future hybrid analysis.

Run this notebook first. It writes `data/train.parquet` and `data/test.parquet`; subsequent experiments must use these files without splitting again. Parquet reading and writing require `pyarrow` in the notebook environment (`%pip install pyarrow` if needed). Re-running preparation overwrites both files, so rerun downstream experiments after changing the data preparation settings.


In [1]:
from pathlib import Path
import pandas as pd

RANDOM_STATE = 18
SAMPLES_PER_CLASS = 5000  # Keep the first experiment affordable; applied after raw deduplication.
TEST_SIZE = 0.2
CV_FOLDS = 5
N_JOBS = 2  # Limit concurrent fits to control memory use.
LABELS = {"not_recommended": 0, "recommended": 1}
LABEL_NAMES = {value: key for key, value in LABELS.items()}

PROJECT_ROOT = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "data" / "digikala-comments.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the project root or one of its subdirectories.")

# Load only the columns needed for this experiment.
df = pd.read_csv(
    PROJECT_ROOT / "data" / "digikala-comments.csv",
    usecols=["body", "recommendation_status"],
)
df["source_row"] = df.index
print("Source rows:", len(df))
display(df["recommendation_status"].value_counts(dropna=False))

Source rows: 6156289


recommendation_status
recommended        4162839
NaN                 894426
no_idea             594121
not_recommended     504903
Name: count, dtype: int64

## 1. Load and audit the data

Remove blank comments and unsupported labels. Keep one copy of identical texts with a consistent label; remove every copy when identical texts have conflicting labels. Report the changes before sampling.


In [2]:
no_idea_mask = df["recommendation_status"].eq("no_idea")
print("Excluded no_idea rows:", no_idea_mask.sum())
df = df.loc[~no_idea_mask].copy()
df["recommendation_status"] = df["recommendation_status"].map(LABELS)
print("Missing or unsupported labels:", df["recommendation_status"].isna().sum())
print("Missing bodies:", df["body"].isna().sum())
df = df.dropna(subset=["body", "recommendation_status"]).copy()
df["body"] = df["body"].str.strip()
print("Blank bodies:", df["body"].eq("").sum())
df = df.loc[df["body"].ne("")].copy()
df["recommendation_status"] = df["recommendation_status"].astype(int)


def audit_and_deduplicate(frame, text_column, stage):
    duplicate_mask = frame.duplicated(text_column, keep=False)
    label_counts = (
        frame[[text_column, "recommendation_status"]]
        .drop_duplicates()
        [text_column].value_counts()
    )
    conflicting_texts = label_counts.index[label_counts.gt(1)]
    conflict_mask = frame[text_column].isin(conflicting_texts)
    consistent = frame.loc[~conflict_mask]
    cleaned = consistent.drop_duplicates(text_column, keep="first").copy()
    report = pd.Series({
        "input_rows": len(frame),
        "rows_in_duplicate_groups": int(duplicate_mask.sum()),
        "duplicate_text_groups": int(frame.loc[duplicate_mask, text_column].nunique()),
        "conflicting_text_groups": len(conflicting_texts),
        "conflicting_rows_removed": int(conflict_mask.sum()),
        "same_label_copies_removed": len(consistent) - len(cleaned),
        "remaining_rows": len(cleaned),
    }, name=stage)
    display(report)
    return cleaned, report


df, raw_duplicate_report = audit_and_deduplicate(df, "body", "full_dataset_raw_text")
display(df["recommendation_status"].map(LABEL_NAMES).value_counts())

Excluded no_idea rows: 594121
Missing or unsupported labels: 894426
Missing bodies: 594
Blank bodies: 0


input_rows                   4667284
rows_in_duplicate_groups     1735483
duplicate_text_groups         153274
conflicting_text_groups         6479
conflicting_rows_removed      848434
same_label_copies_removed     740254
remaining_rows               3078596
Name: full_dataset_raw_text, dtype: int64

recommendation_status
recommended        2642629
not_recommended     435967
Name: count, dtype: int64

### Reproducible sample

Sample up to 5,000 unique comments from each class using the fixed seed. This balances the experiment approximately; the resulting class distribution does not represent the original dataset.


In [3]:
sampled = pd.concat([
    df.loc[df["recommendation_status"].eq(label)].sample(
        n=min(SAMPLES_PER_CLASS, int(df["recommendation_status"].eq(label).sum())),
        random_state=RANDOM_STATE,
    )
    for label in LABELS.values()
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
# Release the full dataset before NLP preprocessing and cross-validation.
del df
df = sampled
del sampled
print("Sampled rows:", len(df))
display(df["recommendation_status"].map(LABEL_NAMES).value_counts())

Sampled rows: 10000


recommendation_status
not_recommended    5000
recommended        5000
Name: count, dtype: int64

### Text preparation and duplicate check

For TF-IDF, normalize and tokenize with Hazm. Keep known verb forms unchanged to preserve negation; lemmatize other tokens. Retain stopwords, one-letter tokens, and half-spaces; emoji are not explicitly removed. Remove empty texts and collisions introduced by preprocessing before splitting the data. The TF-IDF vocabulary is learned inside each CV training fold.


In [4]:
import string
import hazm

normalizer = hazm.Normalizer()
word_tokenizer = hazm.WordTokenizer()
lemmatizer = hazm.Lemmatizer()


def remove_chars(text, chars):
    for char in chars:
        text = text.replace(char, " ")
    return text


def preprocess(text):
    if not isinstance(text, str):
        return None
    text = remove_chars(text.lower(), string.punctuation + "؟،")
    text = normalizer.normalize(text).replace("آ", "ا")
    # Preserve the half-space so negated verbs remain intact during tokenization.
    text = remove_chars(text, "۰۱۲۳۴۵۶۷۸۹")
    tokens = [
        token if token in lemmatizer.verbs
        else lemmatizer.lemmatize(token).split("#")[0]
        for token in word_tokenizer.tokenize(text)
    ]
    return " ".join(tokens) if tokens else None


df["processed_body"] = df["body"].apply(preprocess)
print("Empty after preprocessing:", df["processed_body"].isna().sum())
df = df.dropna(subset=["processed_body"]).copy()
df, processed_duplicate_report = audit_and_deduplicate(
    df, "processed_body", "sample_processed_text"
)
df = df.reset_index(drop=True)
duplicate_report = pd.concat([raw_duplicate_report, processed_duplicate_report], axis=1)
display(df[["body", "processed_body", "recommendation_status"]].head())

Empty after preprocessing: 1


input_rows                   9999
rows_in_duplicate_groups        8
duplicate_text_groups           3
conflicting_text_groups         0
conflicting_rows_removed        0
same_label_copies_removed       5
remaining_rows               9994
Name: sample_processed_text, dtype: int64

,body,processed_body,recommendation_status
0,نصف موهام ریخت همون لطیفس انگار فقط بسته بندیش...,نصف موه ریخت همون لطیفس انگار فقط بسته بندیشو ...,0
1,زبر بود کاش سافت داشت,زبر بود کاش ساف داشت,1
2,قشنگ و اندازه خوب,قشنگ و اندازه خوب,1
3,سلام دماسنج خوبیه و دقت خوبی داره البته من برا...,سلام دماسنج خوبیه و دقت خوبی داره البته من برا...,1
4,بوشو دوست نداشتم و اینکه چرب میکنه موها رو,بوشو دوست نداشتم و اینکه چرب میکنه مو رو,0


## 2. Freeze the train/test split

Reserve 20% of the cleaned sample for test and use five stratified folds on train. All three experiments below use these same rows and labels. This text-level split does not measure generalization to unseen products or future comments.


In [5]:
from sklearn.model_selection import train_test_split, StratifiedKFold

if set(df["recommendation_status"]) != set(LABELS.values()):
    raise ValueError("Both recommendation classes must remain after cleaning.")

X_train, X_test, y_train, y_test = train_test_split(
    df["processed_body"], df["recommendation_status"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["recommendation_status"],
)
assert set(X_train).isdisjoint(set(X_test)), "Duplicate text crosses train/test."
if y_train.value_counts().min() < CV_FOLDS:
    raise ValueError("Not enough training examples per class for cross-validation.")

split_counts = pd.DataFrame({
    "train": y_train.value_counts(),
    "test": y_test.value_counts(),
}).reindex(list(LABELS.values())).rename(index=LABEL_NAMES)
display(split_counts)
print("Train:", len(X_train), "| Reserved test:", len(X_test))

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

,train,test
recommendation_status,,
not_recommended,4000,1000
recommended,3995,999


Train: 7995 | Reserved test: 1999


### Save the shared datasets and CV membership

Keep `source_row`, the original `body`, TF-IDF `processed_body`, and `recommendation_status`. Preserve the selected row order. The train-only `fold` column identifies the validation fold for each row; all other train folds are used for fitting. Test rows never participate in CV.


In [6]:
train_data = df.loc[X_train.index, [
    "source_row", "body", "processed_body", "recommendation_status"
]].copy().reset_index(drop=True)
test_data = df.loc[X_test.index, [
    "source_row", "body", "processed_body", "recommendation_status"
]].copy().reset_index(drop=True)

train_data["fold"] = -1
for fold_id, (_, valid_positions) in enumerate(cv.split(X_train, y_train)):
    train_data.loc[valid_positions, "fold"] = fold_id

assert train_data["fold"].ge(0).all()
assert train_data["source_row"].is_unique and test_data["source_row"].is_unique
assert set(train_data["source_row"]).isdisjoint(test_data["source_row"])

train_path = PROJECT_ROOT / "data" / "train.parquet"
test_path = PROJECT_ROOT / "data" / "test.parquet"
train_data.to_parquet(train_path, engine="pyarrow", index=False)
test_data.to_parquet(test_path, engine="pyarrow", index=False)
print(f"Saved {len(train_data)} train rows to {train_path}")
print(f"Saved {len(test_data)} test rows to {test_path}")
display(pd.crosstab(train_data["fold"], train_data["recommendation_status"]))


Saved 7995 train rows to f:\Roham\projects\Web\Django\vazheh-kav\data\train.parquet
Saved 1999 test rows to f:\Roham\projects\Web\Django\vazheh-kav\data\test.parquet


recommendation_status,0,1
fold,,
0,800,799
1,800,799
2,800,799
3,800,799
4,800,799
